# Wildlife Re-ID — Submission #1: MiewID baseline (open-set)

Run top to bottom. Sections: **0** config · **1** data · **2** EDA · **3** metric + open-set folds · **4** retrieval core ·
**5** CV search · **6** final fit + `submission_v1.csv` with sanity checks.

Rules reminder: only numpy / pandas / scikit-learn (+ allowed libs), species never mixed, no pretrained models or extra data.

## 0. Config

In [1]:
import os, json, time
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.decomposition import PCA

DATA_DIR  = os.environ.get("DATA_DIR", r"C:\Users\Mayank\Downloads\projectwild3\reid_student_data_main")   # folder with train.csv, test.csv, miewid_*.{npy,csv}, sample_submission.csv
NEW       = "new_individual"
SPECIES   = ["hyena", "leopard"]
N_FOLDS   = 5
SEED      = 42
NEW_SHARE = 0.30          # target share of new_individual among validation images (real test rate unknown -> calibrate on LB)
FOLDS_PATH = "folds.csv"  # saved once; reuse in ALL future notebooks so CV stays identical
OUT_PATH   = "submission_v1.csv"

## 1. Load data

In [2]:
def load_data(d=DATA_DIR):
    """Returns train df, test df, MiewID rows aligned to train, MiewID rows aligned to test."""
    tr = pd.read_csv(f"{d}/train.csv")
    te = pd.read_csv(f"{d}/test.csv")
    ids = pd.read_csv(f"{d}/miewid_ids.csv")
    col = "id" if "id" in ids.columns else ids.columns[0]
    F = np.load(f"{d}/miewid_features.npy").astype(np.float32)
    pos = pd.Series(np.arange(len(ids)), index=ids[col].values)
    Xtr, Xte = F[pos.loc[tr.id].values], F[pos.loc[te.id].values]   # KeyError if ids misalign
    tr["viewpoint"] = tr["viewpoint"].fillna("unk")
    te["viewpoint"] = te["viewpoint"].fillna("unk")
    return tr, te, Xtr, Xte



tr, te, Xtr, Xte = load_data()
print(tr.shape, te.shape, Xtr.shape, Xte.shape)

(7163, 4) (2791, 3) (7163, 2152) (2791, 2152)


## 2. EDA & integrity checks

In [3]:
raw_tr = pd.read_csv(f"{DATA_DIR}/train.csv"); raw_te = pd.read_csv(f"{DATA_DIR}/test.csv")
ids_df = pd.read_csv(f"{DATA_DIR}/miewid_ids.csv"); F = np.load(f"{DATA_DIR}/miewid_features.npy")
idcol = "id" if "id" in ids_df.columns else ids_df.columns[0]

print("=== SPECIES ===")
print(pd.concat([raw_tr.species.value_counts().rename("train_imgs"), raw_te.species.value_counts().rename("test_imgs")], axis=1))
print("unique identities (train):"); print(raw_tr.groupby("species").identity.nunique())

print("\n=== IMAGES PER IDENTITY ===")
cnt = raw_tr.groupby(["species", "identity"]).size().rename("n").reset_index()
cnt["bucket"] = pd.cut(cnt.n, [0, 1, 2, 5, 10, 20, 10_000], labels=["1 (singleton)", "2", "3-5", "6-10", "11-20", ">20"])
display(cnt.pivot_table(index="bucket", columns="species", values="n", aggfunc="count", observed=False)) if "display" in globals() else print(cnt.pivot_table(index="bucket", columns="species", values="n", aggfunc="count", observed=False))
for sp, g in cnt.groupby("species"):
    print(f"{sp}: singleton identities {(g.n==1).sum()} ({(g.n==1).mean():.1%}), they hold {(g.n==1).sum()/g.n.sum():.1%} of train images; "
          f"median imgs/identity {g.n.median():.0f}, max {g.n.max()}")

print("\n=== VIEWPOINT (missing) ===")
print("train:", raw_tr.viewpoint.isna().sum(), f"({raw_tr.viewpoint.isna().mean():.2%})", "| test:", raw_te.viewpoint.isna().sum(), f"({raw_te.viewpoint.isna().mean():.2%})")

print("\n=== ALIGNMENT ===")
s_all, s_ids = set(raw_tr.id) | set(raw_te.id), set(ids_df[idcol])
print("rows match:", len(ids_df) == len(F), "| id sets equal:", s_all == s_ids, "| dup ids:", ids_df[idcol].duplicated().sum(), "| train∩test:", len(set(raw_tr.id) & set(raw_te.id)))

print("\n=== FEATURE HEALTH ===")
nrm = np.linalg.norm(F, axis=1)
print("NaN:", int(np.isnan(F).sum()), "| zero-norm rows:", int((nrm < 1e-8).sum()), "| norm min/mean/max:", nrm.min(), nrm.mean(), nrm.max())

print("\n=== 1-NN identity agreement inside train (images with a sibling) ===")
for sp in SPECIES:
    m = (tr.species == sp).values
    X = Xtr[m] / np.linalg.norm(Xtr[m], axis=1, keepdims=True); y = tr.identity.values[m]
    S = X @ X.T; np.fill_diagonal(S, -np.inf)
    has_mate = pd.Series(y).map(pd.Series(y).value_counts()).values > 1
    print(f"{sp}: {(y[S.argmax(1)] == y)[has_mate].mean():.3f}")

=== SPECIES ===
         train_imgs  test_imgs
species                       
leopard        5251       1574
hyena          1912       1217
unique identities (train):
species
hyena      205
leopard    345
Name: identity, dtype: int64

=== IMAGES PER IDENTITY ===
species        hyena  leopard
bucket                       
1 (singleton)      3       99
2                  6       51
3-5               81       87
6-10              69       35
11-20             27       30
>20               19       43
hyena: singleton identities 3 (1.5%), they hold 0.2% of train images; median imgs/identity 6, max 73
leopard: singleton identities 99 (28.7%), they hold 1.9% of train images; median imgs/identity 3, max 391

=== VIEWPOINT (missing) ===
train: 136 (1.90%) | test: 42 (1.50%)

=== ALIGNMENT ===
rows match: True | id sets equal: True | dup ids: 0 | train∩test: 0

=== FEATURE HEALTH ===
NaN: 0 | zero-norm rows: 0 | norm min/mean/max: 48.09179 68.02694 239.04512

=== 1-NN identity agreement inside 

## 3. Metric and open-set CV folds
Each identity gets a random "new-group"; in fold *k* identities from group *k* are held out entirely (`val_new`, truth = `new_individual`)
until they make up ~`NEW_SHARE` of that species' validation images. Multi-image identities also lose ~1/5 of their images per fold (`val_known`).

In [4]:
def map_at_5(y_true, y_pred_lists):
    """Exact competition metric. y_true: list of labels; y_pred_lists: list of ranked label lists."""
    tot = 0.0
    for t, p in zip(y_true, y_pred_lists):
        p = list(dict.fromkeys(p))[:5]          # drop repeats, keep first 5
        if t in p:
            tot += 1.0 / (p.index(t) + 1)
    return tot / len(y_true)


def make_folds(tr, n_folds=N_FOLDS, seed=SEED, new_share=NEW_SHARE):
    """Open-set CV. For every fold k each train image gets a role:
         'val_known' : identity has >=2 images; this image is held out, its siblings stay in train
         'val_new'   : identity entirely held out -> ground truth = new_individual
         'train'     : gallery
       Every identity gets a random 'new-group' (stratified by species); in fold k, identities from group k are
       taken as new until new images make up ~`new_share` of that species' validation images (so one huge identity
       cannot swamp the fold). Singletons are never 'val_known' (no sibling would remain).
    """
    rng = np.random.RandomState(seed)
    tr = tr.reset_index(drop=True)
    group, imgs, role = {}, {}, np.full((len(tr), n_folds), "train", dtype=object)
    for sp in sorted(tr.species.unique()):
        idents = np.array(sorted(tr.loc[tr.species == sp, "identity"].unique()))
        rng.shuffle(idents)
        group.update({i: n % n_folds for n, i in enumerate(idents)})
    for ident, g in tr.groupby("identity", sort=True):           # pass 1: held-out images of known identities
        idx = g.index.values.copy(); rng.shuffle(idx); imgs[ident] = idx
        off = rng.randint(n_folds)
        if len(idx) >= 2:
            for r, i in enumerate(idx):
                role[i, (off + r) % n_folds] = "val_known"
    sp_of = tr.drop_duplicates("identity").set_index("identity").species
    for k in range(n_folds):                                     # pass 2: choose new identities per species
        for sp in sorted(tr.species.unique()):
            sp_idx = np.flatnonzero((tr.species == sp).values)
            n_known = int((role[sp_idx, k] == "val_known").sum())
            target = new_share / (1 - new_share) * n_known
            cands = sorted(i for i, gk in group.items() if gk == k and sp_of[i] == sp)
            rng.shuffle(cands)
            got = 0
            for ident in cands:
                if got >= target:
                    break
                role[imgs[ident], k] = "val_new"
                got += len(imgs[ident])
    out = tr[["id", "species", "identity"]].copy()
    for k in range(n_folds):
        out[f"role_f{k}"] = role[:, k]
    return out


def load_or_make_folds(tr, path=FOLDS_PATH):
    if os.path.exists(path):
        f = pd.read_csv(path)
        assert (f["id"].values == tr["id"].values).all(), "folds.csv does not match train.csv - delete it to rebuild"
        return f
    f = make_folds(tr)
    f.to_csv(path, index=False)
    return f


# self-test of the metric: worked example from the problem statement
yt = ["LEO0317", "HYE0042", NEW, "HYE0011"]
yp = [["LEO0317", "LEO0102"], ["HYE0007", "HYE0042", NEW], ["LEO0102", "LEO0009", NEW], [NEW, "HYE0198"]]
assert abs(map_at_5(yt, yp) - (1 + .5 + 1/3) / 4) < 1e-9
print("map_at_5 OK:", round(map_at_5(yt, yp), 4))

folds = load_or_make_folds(tr)
for k in range(N_FOLDS):
    r = folds[f"role_f{k}"]; v = folds[r != "train"]
    print(f"fold {k}: train={(r=='train').sum()}  val={len(v)}  new share={(v[f'role_f{k}']=='val_new').mean():.1%}  "
          + "  ".join(f"{sp}: {(v.species==sp).sum()}" for sp in SPECIES))

map_at_5 OK: 0.4583
fold 0: train=5127  val=2036  new share=37.6%  hyena: 525  leopard: 1511
fold 1: train=5238  val=1925  new share=32.7%  hyena: 532  leopard: 1393
fold 2: train=4985  val=2178  new share=44.2%  hyena: 549  leopard: 1629
fold 3: train=5326  val=1837  new share=29.2%  hyena: 509  leopard: 1328
fold 4: train=5255  val=1908  new share=32.2%  hyena: 517  leopard: 1391


## 4. Retrieval core
preprocess → DBA (gallery) → QE (query) → cosine → viewpoint bonus → identity aggregation → top-5 → insert `new_individual` at threshold τ.

In [5]:
def l2n(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)


def species_embeddings(Xtr_sp, Xte_sp, cfg):
    """Unsupervised preprocessing fitted on ALL rows of ONE species (train+test features, no labels)."""
    X = l2n(np.vstack([Xtr_sp, Xte_sp]).astype(np.float32))
    if cfg["center"]:
        X = l2n(X - X.mean(0, keepdims=True))
    if cfg["pca"]:
        X = l2n(PCA(cfg["pca"], whiten=True, random_state=0).fit_transform(X).astype(np.float32))
    return X[:len(Xtr_sp)], X[len(Xtr_sp):]


def expand(Q, G, k, alpha=3.0, include_q=True):
    """alpha-weighted query expansion / database augmentation: mix each row with its top-k neighbours in G."""
    S = Q @ G.T
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    w = np.clip(np.take_along_axis(S, idx, 1), 0, None) ** alpha
    out = (w[:, :, None] * G[idx]).sum(1)
    if include_q:
        out = out + Q
    return l2n(out)


def identity_scores(S, gcode, agg):
    """Image-level sims (nq x ng) -> identity-level (nq x n_id). gcode in 0..n_id-1. agg: 'max' | 'top2' | 'top3'..."""
    order = np.argsort(gcode, kind="stable")
    Ss, c = S[:, order], gcode[order]
    starts = np.r_[0, np.flatnonzero(np.diff(c)) + 1]
    ends = np.r_[starts[1:], len(c)]
    if agg == "max":
        return np.maximum.reduceat(Ss, starts, axis=1)
    k = int(agg[3:])
    out = np.empty((S.shape[0], len(starts)), np.float32)
    for j, (a, b) in enumerate(zip(starts, ends)):
        kk = min(k, b - a)
        out[:, j] = np.partition(Ss[:, a:b], -kk, axis=1)[:, -kk:].mean(1)
    return out


def rank_topk(Q, G, gcode, vq, vg, cfg, topn=5):
    """Returns (top identity indices, their scores), both nq x topn, sorted best-first."""
    if cfg["dba"] > 0:
        G = expand(G, G, cfg["dba"], include_q=False)       # self is its own top-1 neighbour
    if cfg["qe"] > 0:
        Q = expand(Q, G, cfg["qe"], include_q=True)
    S = Q @ G.T
    if cfg["vp"] > 0:                                        # missing viewpoints are coded so they never match
        S = S + cfg["vp"] * (vq[:, None] == vg[None, :])
    sc = identity_scores(S, gcode, cfg["agg"])
    idx = np.argsort(-sc, axis=1)[:, :min(topn, sc.shape[1])]
    return idx, np.take_along_axis(sc, idx, 1)


# ---------- open-set threshold logic ----------
def new_position(top_sc, tau):
    """0-based rank at which new_individual is inserted = number of candidates with score >= tau (capped at 4)."""
    return np.minimum((top_sc >= tau).sum(1), 4)


def apply_new(top_names, top_sc, tau):
    pos = new_position(top_sc, tau)
    out = []
    for names, p in zip(top_names, pos):
        lst = list(names)
        lst.insert(int(p), NEW)
        out.append(lst[:5])
    return out


def score_samples(top_sc, has, j, isnew, tau):
    """Vectorised per-query AP@5 given tau. has/j: whether/where the true identity is in the top-5 (known queries)."""
    pos = new_position(top_sc, tau)
    r = np.where(isnew, pos, j + (pos <= j))
    ok = (isnew | has) & (r < 5)
    return np.where(ok, 1.0 / (r + 1), 0.0)


def tune_tau(top_sc, has, j, isnew):
    q = np.quantile(top_sc.ravel(), np.linspace(0, 1, 401))
    grid = np.r_[-np.inf, np.unique(q), np.inf]
    maps = np.array([score_samples(top_sc, has, j, isnew, t).mean() for t in grid])
    b = int(maps.argmax())
    return float(grid[b]), float(maps[b]), float(maps[0])   # tau, best MAP, MAP with 'new' always at rank 5

## 5. Open-set CV + greedy config search
A change is kept only if pooled OOF MAP@5 improves by > 1e-4. τ is tuned per species on pooled out-of-fold scores.

In [6]:
vp_map = {v: i for i, v in enumerate(sorted(set(tr.viewpoint) | set(te.viewpoint)))}
def vp_codes(df):  # missing viewpoint -> -1
    return np.where(df.viewpoint.values == "unk", -1, df.viewpoint.map(vp_map).values)
vp_tr, vp_te = vp_codes(tr), vp_codes(te)
gal_vp = lambda v: np.where(v < 0, -2, v)      # gallery 'unknown' != query 'unknown' -> never matches

_cache = {}
def get_pre(sp, cfg):
    key = (sp, cfg["center"], cfg["pca"])
    if key not in _cache:
        m_tr, m_te = (tr.species == sp).values, (te.species == sp).values
        _cache[key] = species_embeddings(Xtr[m_tr], Xte[m_te], cfg)[0]
    return _cache[key]

def run_cfg(cfg):
    out, allsc = {"species": {}}, []
    for sp in SPECIES:
        P = get_pre(sp, cfg)
        sub = np.flatnonzero((tr.species == sp).values)
        loc = np.full(len(tr), -1); loc[sub] = np.arange(len(sub))
        T, H, J, N, FD = [], [], [], [], []
        for k in range(N_FOLDS):
            role = folds[f"role_f{k}"].values
            gi, vi = sub[role[sub] == "train"], sub[role[sub] != "train"]
            classes, gcode = np.unique(tr.identity.values[gi], return_inverse=True)
            top, sc = rank_topk(P[loc[vi]], P[loc[gi]], gcode, vp_tr[vi], gal_vp(vp_tr[gi]), cfg)
            isnew = role[vi] == "val_new"
            cm = {c: i for i, c in enumerate(classes)}
            y = np.array([-1 if n else cm[i] for i, n in zip(tr.identity.values[vi], isnew)])
            match = top == y[:, None]
            T.append(sc); H.append(match.any(1)); J.append(match.argmax(1)); N.append(isnew); FD.append(np.full(len(vi), k))
        T, H, J, N, FD = map(np.concatenate, (T, H, J, N, FD))
        tau, m, m_no = tune_tau(T, H, J, N)
        s = score_samples(T, H, J, N, tau)
        out["species"][sp] = dict(tau=tau, map=m, n=len(T), new1=float((new_position(T, tau) == 0).mean()), true_new=float(N.mean()),
                                  fold_maps=[float(s[FD == k].mean()) for k in range(N_FOLDS)], map_new_last=m_no, recall5=float(H[~N].mean()))
        allsc.append(s)
    out["map"] = float(np.concatenate(allsc).mean())
    return out

def show(tag, r):
    print(f"{tag:<34} CV MAP@5={r['map']:.4f} | " + " | ".join(f"{sp}: {v['map']:.4f} (tau={v['tau']:.3f})" for sp, v in r["species"].items()))

In [7]:
t0 = time.time()
cur = dict(agg="max", center=False, pca=0, dba=0, qe=0, vp=0.0)
best = run_cfg(cur); show("BASELINE " + str(cur), best)
stages = [("agg", ["top2", "top3"]), ("center", [True]), ("pca", [256, 512]), ("dba", [2, 3]), ("qe", [2, 3]), ("vp", [0.02, 0.05, 0.1])]
for key, vals in stages:
    for v in vals:
        c = {**cur, key: v}; r = run_cfg(c); show(f"  try {key}={v}", r)
        if r["map"] > best["map"] + 1e-4:
            cur, best = c, r; print(f"  -> KEEP {key}={v}")

print("\n================ FINAL CV ================\nconfig:", cur)
for sp, v in best["species"].items():
    print(f"{sp}: n_val={v['n']}  MAP@5={v['map']:.4f}  true new rate={v['true_new']:.1%}  predicted new@1={v['new1']:.1%}  "
          f"known-recall@5={v['recall5']:.3f}  MAP(new always last)={v['map_new_last']:.4f}")
    print(f"   per-fold MAP: {np.round(v['fold_maps'], 4).tolist()} (std {np.std(v['fold_maps']):.4f})")
print(f"OVERALL CV MAP@5 = {best['map']:.4f}   [{time.time()-t0:.0f}s]")
TAUS = {sp: v["tau"] for sp, v in best["species"].items()}
json.dump(dict(cfg=cur, tau=TAUS, cv_map=best["map"]), open("best_config.json", "w"), indent=2)

BASELINE {'agg': 'max', 'center': False, 'pca': 0, 'dba': 0, 'qe': 0, 'vp': 0.0} CV MAP@5=0.8378 | hyena: 0.8537 (tau=0.498) | leopard: 0.8320 (tau=0.600)
  try agg=top2                     CV MAP@5=0.8259 | hyena: 0.8389 (tau=0.464) | leopard: 0.8212 (tau=0.572)
  try agg=top3                     CV MAP@5=0.8137 | hyena: 0.8167 (tau=0.453) | leopard: 0.8126 (tau=0.549)
  try center=True                  CV MAP@5=0.8435 | hyena: 0.8475 (tau=0.402) | leopard: 0.8421 (tau=0.489)
  -> KEEP center=True
  try pca=256                      CV MAP@5=0.8695 | hyena: 0.8697 (tau=0.306) | leopard: 0.8694 (tau=0.398)
  -> KEEP pca=256
  try pca=512                      CV MAP@5=0.8793 | hyena: 0.8782 (tau=0.194) | leopard: 0.8797 (tau=0.291)
  -> KEEP pca=512
  try dba=2                        CV MAP@5=0.8792 | hyena: 0.8782 (tau=0.195) | leopard: 0.8796 (tau=0.287)
  try dba=3                        CV MAP@5=0.8797 | hyena: 0.8791 (tau=0.180) | leopard: 0.8800 (tau=0.283)
  -> KEEP dba=3
  try qe

## 6. Final fit on full train → `submission_v1.csv`

In [8]:
cfg = cur
preds = pd.Series(index=te.index, dtype=object)
for sp in SPECIES:                                           # species never mixed
    m_tr, m_te = (tr.species == sp).values, (te.species == sp).values
    G, Q = species_embeddings(Xtr[m_tr], Xte[m_te], cfg)
    classes, gcode = np.unique(tr.identity.values[m_tr], return_inverse=True)
    top, sc = rank_topk(Q, G, gcode, vp_te[m_te], gal_vp(vp_tr[m_tr]), cfg)
    preds[te.index[m_te]] = [" ".join(l) for l in apply_new(classes[top], sc, TAUS[sp])]
sub = pd.DataFrame({"id": te.id.values, "predictions": preds.values})

# ---- sanity checks ----
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert len(sub) == len(ss) == len(te), "row count"
if len(sub) != 2791: print("WARNING: expected 2791 rows, got", len(sub))
assert set(sub.id) == set(ss.id) and sub.id.is_unique, "ids differ from sample_submission"
assert not sub.predictions.isna().any(), "NaN predictions"
ident_species = tr.drop_duplicates("identity").set_index("identity").species.to_dict()
test_species = te.set_index("id").species.to_dict()
new_rank = Counter()
for i, p in zip(sub.id, sub.predictions):
    labs = p.split()
    assert 1 <= len(labs) <= 5 and len(set(labs)) == len(labs), f"bad label count/dupes in {i}"
    for l in labs:
        assert l == NEW or (l in ident_species and ident_species[l] == test_species[i]), f"invalid label {l} for {i}"
    new_rank[labs.index(NEW) + 1 if NEW in labs else 0] += 1
print("ALL SANITY CHECKS PASSED")
print("new_individual rank distribution (0 = absent):", dict(sorted(new_rank.items())))
print(f"predicted new@1 rate: {new_rank[1] / len(sub):.1%}   (compare with CV true new rate above)")
print(sub.sample(10, random_state=0).to_string(index=False))
sub.to_csv(OUT_PATH, index=False); print("saved", OUT_PATH)

ALL SANITY CHECKS PASSED
new_individual rank distribution (0 = absent): {1: 1185, 2: 1394, 3: 157, 4: 33, 5: 22}
predicted new@1 rate: 42.5%   (compare with CV true new rate above)
      id                                    predictions
te_02465 HYE0236 new_individual HYE0179 HYE0199 HYE0346
te_00689 new_individual LEO0486 LEO0549 LEO0518 LEO0508
te_02684 new_individual HYE0675 HYE0512 HYE0126 HYE0616
te_00992 HYE0524 new_individual HYE0034 HYE0095 HYE0543
te_00617 HYE0188 new_individual HYE0222 HYE0214 HYE0125
te_00228 new_individual LEO0458 LEO0546 LEO0409 LEO0156
te_02614 new_individual LEO0256 LEO0434 LEO0333 LEO0229
te_01666 new_individual HYE0123 HYE0636 HYE0281 HYE0353
te_00321 HYE0425 new_individual HYE0535 HYE0111 HYE0031
te_01513 HYE0177 new_individual HYE0603 HYE0446 HYE0376
saved submission_v1.csv
